# DocuMind AI: RAG & LangGraph Architecture Exploration

Welcome to the **DocuMind AI** experimentation notebook!

This notebook breaks down the core concepts behind **Retrieval-Augmented Generation (RAG)** step-by-step:
1. **Document Loading**: Extracting text and page metadata from PDFs.
2. **Text Chunking**: Splitting text into overlapping semantic passages.
3. **Embeddings & Vector Spaces**: Translating text into high-dimensional vectors.
4. **FAISS Indexing**: Storing and querying vector spaces.
5. **Semantic Retrieval**: Finding nearest neighbors to user queries.
6. **LangGraph Orchestration**: State-driven workflow routing.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is accessible
ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Project Root:", ROOT_DIR)

## 1. Document Loading
We load our sample multi-page AI document using `src.document_loader.load_single_pdf`.

In [ ]:
from src.document_loader import load_single_pdf

pdf_path = ROOT_DIR / "data" / "documents" / "sample_ai_notes.pdf"
pages = load_single_pdf(pdf_path)

print(f"Total Pages Loaded: {len(pages)}\n")
for i, page in enumerate(pages):
    print(f"Page {i+1} Metadata: {page.metadata}")
    print(f"Page {i+1} Preview: {page.page_content.strip()[:120]}...\n")

## 2. Text Chunking
We split the loaded pages into overlapping chunks using `src.chunker.split_documents`.

In [ ]:
from src.chunker import split_documents

chunks = split_documents(pages, chunk_size=500, chunk_overlap=100)
print(f"Produced {len(chunks)} chunks from {len(pages)} pages.\n")

# Inspect first chunk
print("Chunk 0 Metadata:", chunks[0].metadata)
print("Chunk 0 Text:\n", chunks[0].page_content)

## 3. Offline Vector Embeddings & FAISS Indexing
We test creating a FAISS vector store using deterministic test embeddings to visualize the index mechanics.

In [ ]:
from src.embeddings import get_embedding_model
from src.vector_store import create_vector_store

# Use offline fake embeddings for instant local demonstration without API consumption
embedding_model = get_embedding_model(provider="fake")
test_vector_dir = ROOT_DIR / "vectorstore" / "notebook_faiss_index"

vector_store = create_vector_store(chunks, embedding_model, save_path=test_vector_dir)
print("FAISS vector store successfully created and saved!")

## 4. Semantic Similarity Search
Now we query the FAISS index for relevant documents and observe distance scores.

In [ ]:
from src.retriever import retrieve_relevant_chunks
from src.utils import format_sources

query = "What is Retrieval-Augmented Generation?"
results = retrieve_relevant_chunks(vector_store, query=query, top_k=3)

print(f"Retrieved {len(results)} chunks for query: '{query}'\n")
for doc, score in results:
    print(f"Source: {doc.metadata['source']} | Page: {doc.metadata['page'] + 1} | Score: {score:.4f}")
    print(f"Content: {doc.page_content.strip()[:150]}...\n")

sources = format_sources([doc for doc, _ in results])
print("Formatted Sources:", [s['display_text'] for s in sources])

## 5. LangGraph Architecture
In DocuMind AI, LangGraph coordinates the stateful workflow:
```
START -> Contextualize -> Retrieve -> Check Relevance -> Generate / Fallback -> END
```
This architecture ensures responses are grounded, source-attributed, and hallucination-free!